# 🎙️ VoxCPM2 — One Click Google Colab

Select a GPU runtime and click **Runtime → Run all**. The launcher bootstraps Python 3.12 automatically when needed, installs compatible dependencies, downloads VoxCPM2, and starts Gradio.

The actual VoxCPM2 runtime uses Python 3.12, NumPy 1.26.4, VoxCPM 2.0.3, and Gradio 6.x.


In [ ]:
#@title 🚀 VoxCPM2 — One Click Colab Setup & Launch
#
# This notebook is designed for:
#   1) Fresh Google Colab
#   2) T4 GPU
#   3) User presses "Run all"
#
# It does NOT assume that Colab's current kernel Python is the model's
# supported Python. If the active kernel is Python 3.13, it creates a
# separate Python 3.12 environment with uv and runs the actual app there.

import os
import sys
import subprocess
import textwrap
from pathlib import Path

APP_PATH = Path("/content/voxcpm2_app.py")
VENV_PATH = Path("/content/voxcpm2_py312")
MODEL_CACHE = Path("/content/hf_cache")

MODEL_CACHE.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("🎙️ VoxCPM2 — One Click Colab Launcher")
print("=" * 72)
print("Current notebook Python:", sys.version.split()[0])

# Check GPU before doing expensive package installation.
try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU detected. In Colab choose Runtime → Change runtime type "
            "→ Hardware accelerator → T4 GPU, then Run all again."
        )
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except Exception as e:
    print("GPU check:", e)
    raise

# Write the complete application to disk.
APP_PATH.write_text('\nimport os\nimport sys\nimport signal\nimport gc\nimport random\nimport tempfile\nimport subprocess\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nimport gradio as gr\nfrom voxcpm import VoxCPM\n\nMODEL_ID = "openbmb/VoxCPM2"\nSAMPLE_RATE = None\n\nprint("=" * 70)\nprint("VoxCPM2 — Multilingual TTS / Voice Design / Voice Cloning")\nprint("=" * 70)\nprint(f"Python: {sys.version.split()[0]}")\nprint(f"PyTorch: {torch.__version__}")\nprint(f"CUDA available: {torch.cuda.is_available()}")\n\nif not torch.cuda.is_available():\n    raise RuntimeError(\n        "No CUDA GPU detected. In Colab choose Runtime → Change runtime type "\n        "→ Hardware accelerator → T4 GPU (or another NVIDIA GPU)."\n    )\n\nprint(f"GPU: {torch.cuda.get_device_name(0)}")\nprint(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")\n\n# Official VoxCPM2 Python API:\n# model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)\n# wav = model.generate(...)\n# Reference: official VoxCPM project / PyPI.\n\nprint("\\n🚀 Loading VoxCPM2...")\nmodel = VoxCPM.from_pretrained(\n    MODEL_ID,\n    load_denoiser=False,\n    optimize=False,\n)\n\nSAMPLE_RATE = model.tts_model.sample_rate\nprint(f"✅ Model loaded. Sample rate: {SAMPLE_RATE} Hz")\n\n# Lightweight warm-up\nprint("🔥 Warming up...")\nwith torch.inference_mode():\n    warm = model.generate(\n        text="Hello, warm up test.",\n        cfg_value=2.0,\n        inference_timesteps=5,\n        seed=42,\n    )\nwarm = np.asarray(warm, dtype=np.float32).reshape(-1)\nif warm.size == 0:\n    raise RuntimeError("Warm-up returned no audio.")\ndel warm\ngc.collect()\ntorch.cuda.empty_cache()\nprint("✅ Warm-up complete.")\n\nCSS = """\n@import url(\'https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap\');\n* { font-family: \'Inter\', sans-serif !important; }\n.gradio-container { max-width: 1000px !important; margin: auto !important; }\n.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }\n.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }\n.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }\n.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }\n.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }\n.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }\n.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }\n.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }\nbutton.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }\n"""\n\nBRAND_HTML = """\n<div class="brand-header">\n  <div class="brand-title">🎙️ VoxCPM2 — Multilingual TTS</div>\n  <div class="brand-subtitle">Created by <strong>The BlackBox Security</strong> &nbsp;|&nbsp; 2B Parameters · 30 Languages · 48kHz Output · Voice Design &amp; Cloning</div>\n  <div class="btn-row">\n    <a href="https://www.youtube.com/@TheBlackBoxSecurity?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>\n    <a href="https://x.com/TheBlackBoxSecurity" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>\n    <a href="https://TheBlackBoxSecurity.com" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>\n  </div>\n</div>\n"""\n\ndef save_wav(wav):\n    wav = np.asarray(wav, dtype=np.float32).reshape(-1)\n    if wav.size == 0:\n        raise gr.Error("Model returned no audio.")\n    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)\n    tmp.close()\n    sf.write(tmp.name, wav, SAMPLE_RATE)\n    return tmp.name\n\ndef get_seed(seed, locked):\n    try:\n        seed_i = int(seed)\n    except (TypeError, ValueError):\n        seed_i = -1\n\n    if locked and seed_i >= 0:\n        return seed_i\n    return random.randint(0, 2**31 - 1)\n\ndef generate(**kwargs):\n    try:\n        with torch.inference_mode():\n            wav = model.generate(**kwargs)\n        wav = np.asarray(wav, dtype=np.float32).reshape(-1)\n        if wav.size == 0:\n            raise RuntimeError("Model returned empty audio.")\n        return wav\n    except torch.cuda.OutOfMemoryError as e:\n        torch.cuda.empty_cache()\n        raise gr.Error(\n            "GPU out of memory. Try shorter text or fewer inference steps."\n        ) from e\n    except Exception as e:\n        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}") from e\n\ndef tts_generate(text, cfg, steps, seed, locked):\n    if not text or not text.strip():\n        raise gr.Error("Please enter some text.")\n    used = get_seed(seed, locked)\n    wav = generate(\n        text=text.strip(),\n        cfg_value=float(cfg),\n        inference_timesteps=int(steps),\n        seed=used,\n    )\n    return save_wav(wav), used\n\ndef voice_design(description, text, cfg, steps, seed, locked):\n    if not description or not description.strip():\n        raise gr.Error("Please enter a voice description.")\n    if not text or not text.strip():\n        raise gr.Error("Please enter text content.")\n    used = get_seed(seed, locked)\n    wav = generate(\n        text=f"({description.strip()}){text.strip()}",\n        cfg_value=float(cfg),\n        inference_timesteps=int(steps),\n        seed=used,\n    )\n    return save_wav(wav), used\n\ndef voice_clone(text, ref_audio, style, cfg, steps, seed, locked):\n    if not text or not text.strip():\n        raise gr.Error("Please enter text.")\n    if not ref_audio:\n        raise gr.Error("Please upload reference audio.")\n    if style and style.strip():\n        text = f"({style.strip()}){text.strip()}"\n    else:\n        text = text.strip()\n    used = get_seed(seed, locked)\n    wav = generate(\n        text=text,\n        reference_wav_path=ref_audio,\n        cfg_value=float(cfg),\n        inference_timesteps=int(steps),\n        seed=used,\n    )\n    return save_wav(wav), used\n\ndef ultimate_clone(text, ref_audio, transcript, cfg, steps, seed, locked):\n    if not text or not text.strip():\n        raise gr.Error("Please enter text.")\n    if not ref_audio:\n        raise gr.Error("Please upload reference audio.")\n    if not transcript or not transcript.strip():\n        raise gr.Error("Please enter the reference audio transcript.")\n    used = get_seed(seed, locked)\n    wav = generate(\n        text=text.strip(),\n        prompt_wav_path=ref_audio,\n        prompt_text=transcript.strip(),\n        reference_wav_path=ref_audio,\n        cfg_value=float(cfg),\n        inference_timesteps=int(steps),\n        seed=used,\n    )\n    return save_wav(wav), used\n\ndef seed_row():\n    with gr.Row():\n        seed = gr.Number(value=-1, label="Seed", precision=0, scale=3)\n        locked = gr.Checkbox(value=False, label="🔒 Lock Seed", scale=1)\n    return seed, locked\n\nwith gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="VoxCPM2 TTS") as demo:\n    gr.HTML(BRAND_HTML)\n\n    with gr.Tab("🗣️ Text-to-Speech"):\n        with gr.Accordion("📖 Instructions", open=False):\n            gr.Markdown(\n                "Enter any text and the model will synthesize speech with natural prosody.\\n\\n"\n                "- **CFG Scale**: Higher = more adherence to text, lower = more natural/relaxed\\n"\n                "- **Inference Steps**: Higher = better quality but slower (try 6-10 for speed)\\n"\n                "- **Seed**: Shows the seed used. Lock it to reproduce a result.\\n"\n                "- **Supported Languages (30):** Arabic, Burmese, Chinese, Danish, Dutch, English, "\n                "Finnish, French, German, Greek, Hebrew, Hindi, Indonesian, Italian, Japanese, Khmer, "\n                "Korean, Lao, Malay, Norwegian, Polish, Portuguese, Russian, Spanish, Swahili, Swedish, "\n                "Tagalog, Thai, Turkish, Vietnamese"\n            )\n        with gr.Row():\n            with gr.Column():\n                tts_text = gr.Textbox(\n                    label="Text", lines=4,\n                    placeholder="Enter text in any supported language..."\n                )\n                with gr.Row():\n                    tts_cfg = gr.Slider(0.5, 5.0, value=2.0, step=0.1, label="CFG Scale")\n                    tts_steps = gr.Slider(5, 30, value=10, step=1, label="Inference Steps")\n                tts_seed, tts_locked = seed_row()\n                tts_btn = gr.Button("🗣️ Generate Speech", variant="primary", size="lg")\n            with gr.Column():\n                tts_out = gr.Audio(label="Output", type="filepath")\n        tts_btn.click(\n            tts_generate,\n            [tts_text, tts_cfg, tts_steps, tts_seed, tts_locked],\n            [tts_out, tts_seed],\n        )\n\n    with gr.Tab("🎨 Voice Design"):\n        with gr.Accordion("📖 Instructions", open=False):\n            gr.Markdown(\n                "Create a new voice from a natural-language description.\\n\\n"\n                "**Examples:**\\n"\n                "- `A young woman, gentle and sweet voice`\\n"\n                "- `An elderly British man, deep and authoritative`\\n"\n                "- `A calm female narrator with a warm tone`"\n            )\n        with gr.Row():\n            with gr.Column():\n                vd_desc = gr.Textbox(label="Voice Description", lines=2,\n                                     placeholder="A young woman, gentle and sweet voice")\n                vd_text = gr.Textbox(label="Text Content", lines=3,\n                                     placeholder="Hello, welcome to VoxCPM2!")\n                with gr.Row():\n                    vd_cfg = gr.Slider(0.5, 5.0, value=2.0, step=0.1, label="CFG Scale")\n                    vd_steps = gr.Slider(5, 30, value=10, step=1, label="Inference Steps")\n                vd_seed, vd_locked = seed_row()\n                vd_btn = gr.Button("🎨 Design Voice", variant="primary", size="lg")\n            with gr.Column():\n                vd_out = gr.Audio(label="Output", type="filepath")\n        vd_btn.click(\n            voice_design,\n            [vd_desc, vd_text, vd_cfg, vd_steps, vd_seed, vd_locked],\n            [vd_out, vd_seed],\n        )\n\n    with gr.Tab("🎛️ Voice Cloning"):\n        with gr.Accordion("📖 Instructions", open=False):\n            gr.Markdown(\n                "Upload a clear reference clip and synthesize new text in that voice.\\n\\n"\n                "- Reference audio: 5–30 seconds recommended\\n"\n                "- Optional style description can control delivery"\n            )\n        with gr.Row():\n            with gr.Column():\n                vc_ref = gr.Audio(label="Reference Audio", type="filepath")\n                vc_text = gr.Textbox(\n                    label="Text to Synthesize", lines=3,\n                    placeholder="This is a cloned voice generated by VoxCPM2."\n                )\n                vc_style = gr.Textbox(\n                    label="Style Description (optional)", lines=1,\n                    placeholder="slightly faster, cheerful tone"\n                )\n                with gr.Row():\n                    vc_cfg = gr.Slider(0.5, 5.0, value=2.0, step=0.1, label="CFG Scale")\n                    vc_steps = gr.Slider(5, 30, value=10, step=1, label="Inference Steps")\n                vc_seed, vc_locked = seed_row()\n                vc_btn = gr.Button("🎛️ Clone Voice", variant="primary", size="lg")\n            with gr.Column():\n                vc_out = gr.Audio(label="Output", type="filepath")\n        vc_btn.click(\n            voice_clone,\n            [vc_text, vc_ref, vc_style, vc_cfg, vc_steps, vc_seed, vc_locked],\n            [vc_out, vc_seed],\n        )\n\n    with gr.Tab("🎙️ Ultimate Cloning"):\n        with gr.Accordion("📖 Instructions", open=False):\n            gr.Markdown(\n                "Provide reference audio and its exact transcript for maximum fidelity.\\n\\n"\n                "- Both reference audio and transcript are required\\n"\n                "- Use the same reference file for prompt/reference audio"\n            )\n        with gr.Row():\n            with gr.Column():\n                uc_ref = gr.Audio(label="Reference Audio", type="filepath")\n                uc_transcript = gr.Textbox(\n                    label="Reference Audio Transcript", lines=2,\n                    placeholder="The exact transcript of the reference audio."\n                )\n                uc_text = gr.Textbox(\n                    label="Text to Synthesize", lines=3,\n                    placeholder="This is an ultimate cloning demonstration."\n                )\n                with gr.Row():\n                    uc_cfg = gr.Slider(0.5, 5.0, value=2.0, step=0.1, label="CFG Scale")\n                    uc_steps = gr.Slider(5, 30, value=10, step=1, label="Inference Steps")\n                uc_seed, uc_locked = seed_row()\n                uc_btn = gr.Button("🎙️ Ultimate Clone", variant="primary", size="lg")\n            with gr.Column():\n                uc_out = gr.Audio(label="Output", type="filepath")\n        uc_btn.click(\n            ultimate_clone,\n            [uc_text, uc_ref, uc_transcript, uc_cfg, uc_steps, uc_seed, uc_locked],\n            [uc_out, uc_seed],\n        )\n\nprint("\\n🌐 Launching Gradio...")\ndemo.queue().launch(share=True, debug=True)\n', encoding="utf-8")

# Install uv in the notebook environment; it is only used to bootstrap the
# supported Python interpreter and does not matter to the final app.
print("\n📦 Installing uv bootstrap tool...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])

def run(cmd, **kwargs):
    print(">", " ".join(map(str, cmd)))
    return subprocess.run(cmd, check=True, text=True, **kwargs)

# Build a Python 3.12 environment so the user can stay on Colab's current
# notebook kernel even when it is Python 3.13.
if not (VENV_PATH / "bin" / "python").exists():
    print("\n🐍 Creating Python 3.12 environment...")
    run([
        sys.executable, "-m", "uv", "venv",
        "--python", "3.12",
        str(VENV_PATH),
    ])

py = VENV_PATH / "bin" / "python"
pip = VENV_PATH / "bin" / "pip"

print("\n🔧 Installing stable VoxCPM2 runtime packages...")
run([
    str(py), "-m", "pip", "install",
    "-q", "--upgrade", "pip"
])

# PyTorch is installed explicitly for the supported Python 3.12 environment.
# cu128 wheels match the 2026 Colab/PyTorch stack and support T4.
run([
    str(py), "-m", "pip", "install",
    "-q",
    "--index-url", "https://download.pytorch.org/whl/cu128",
    "torch>=2.5,<3",
    "torchaudio>=2.5,<3",
])

# Keep NumPy below 2 to avoid the ABI mismatch that caused the original
# notebook failure.
run([
    str(py), "-m", "pip", "install",
    "-q",
    "numpy==1.26.4",
    "voxcpm==2.0.3",
    "gradio>=6,<7",
    "soundfile",
])

# Verify the important versions inside the actual Python 3.12 environment.
check = subprocess.run(
    [
        str(py), "-c",
        (
            "import sys, numpy, torch; "
            "import voxcpm, gradio; "
            "print('PYTHON=' + sys.version.split()[0]); "
            "print('NUMPY=' + numpy.__version__); "
            "print('TORCH=' + torch.__version__); "
            "print('GRADIO=' + gradio.__version__); "
            "print('CUDA=' + str(torch.cuda.is_available())); "
            "print('GPU=' + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'))"
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)
print(check.stdout)

if "PYTHON=3.12" not in check.stdout:
    raise RuntimeError("Python 3.12 bootstrap failed.")
if "NUMPY=1.26.4" not in check.stdout:
    raise RuntimeError("NumPy 1.26.4 bootstrap failed.")
if "CUDA=True" not in check.stdout:
    raise RuntimeError(
        "The Python 3.12 environment cannot see the Colab GPU. "
        "Restart the runtime and Run all again."
    )

# Run the app in the supported Python 3.12 process.
env = os.environ.copy()
env["HF_HOME"] = str(MODEL_CACHE)
env["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE / "hub")
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

print("=" * 72)
print("✅ Environment ready")
print("🚀 Starting VoxCPM2 in the supported Python 3.12 process...")
print("=" * 72)
print("The first launch downloads the VoxCPM2 model from Hugging Face.")
print("After the model loads, Gradio will print a public https:// link.")
print()

# Blocking run: the public Gradio URL stays alive while this notebook cell runs.
os.execve(str(py), [str(py), str(APP_PATH)], env)
